# Model Calibration and Uncertainty Analysis

## Objective
Assess probabilistic calibration curve, Brier score, and prediction interval uncertainty.

## Input Datasets
- backend/models/flood_risk_model.joblib
- data/processed/mumbai_flood_feature_matrix.csv

---

### Step 1: Brier Score and Calibration Curve Calculation

In [2]:
import os
import json
import joblib
import pandas as pd
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve

base_dir = os.path.abspath(os.path.join(os.getcwd(), "..")) if "notebooks" in os.getcwd() else os.getcwd()
rf = joblib.load(os.path.join(base_dir, "backend", "models", "flood_risk_model.joblib"))
with open(os.path.join(base_dir, "backend", "models", "feature_names.json"), "r") as f:
    features = json.load(f)

df = pd.read_csv(os.path.join(base_dir, "data", "processed", "mumbai_flood_feature_matrix.csv"))
df["date_dt"] = pd.to_datetime(df["date"])
test_df = df[df["date_dt"].dt.year >= 2022]

y_true = test_df["target_high_risk"]
y_prob = rf.predict_proba(test_df[features])[:, 1]

brier = brier_score_loss(y_true, y_prob)
prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=5)

print(f"Brier Score Loss: {round(brier, 4)} (Closer to 0 indicates superior calibration)")
cal_df = pd.DataFrame({"Mean Predicted Prob": prob_pred.round(3), "Empirical True Fraction": prob_true.round(3)})
print("Reliability Table:")
print(cal_df)


Brier Score Loss: 0.0692 (Closer to 0 indicates superior calibration)
Reliability Table:
   Mean Predicted Prob  Empirical True Fraction
0                0.071                    0.015
1                0.278                    0.196
2                0.490                    0.400
3                0.696                    0.784
4                0.961                    0.972


---
## Analytical Limitations & Assumptions
Tree-based probability estimates can be slightly under-confident near extremes; Brier score checks overall reliability.

## Key Conclusions
Model achieves low Brier score (<0.12), validating that predicted probabilities correspond closely to empirical incident likelihoods.